In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms
import os
import random
import numpy as np

# SEED = 0
# torch.manual_seed(SEED)
# os.environ['PYTHONHASHSEED'] = str(SEED)
# random.seed(SEED)
# np.random.seed(SEED)
# g = torch.Generator()
# g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# if torch.cuda.is_available():
#     torch.cuda.manual_seed(SEED)
#     torch.cuda.manual_seed_all(SEED)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False
#     torch.use_deterministic_algorithms(True)
#     os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'


################################
#     RESTART     RUNTIME      #
################################
from ssonn.model.utils import *
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.metrics.train_metrics import *
from ssonn.train.train import *

In [2]:
class SimpleFCN(nn.Module):
    def __init__(self):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(3, 3)

    def forward(self, x):
        x = self.fc0(x)
        return x

In [3]:
model = SimpleFCN()
sparse_model = convert_dense_to_sparse_network(model, layers=[model.fc0], device=device)

In [4]:
sparse_model.fc0.weight_indices, sparse_model.fc0.weight_values

(tensor([[0, 0, 0, 1, 1, 1, 2, 2, 2],
         [0, 1, 2, 0, 1, 2, 0, 1, 2]]),
 Parameter containing:
 tensor([-0.5141, -0.4797, -0.5261,  0.0422,  0.0158,  0.5761,  0.4235,  0.3361,
          0.2489], requires_grad=True))

In [5]:
torch.round(sparse_model.fc0.create_sparse_tensor().to_dense(), decimals=2)

tensor([[-0.5100, -0.4800, -0.5300],
        [ 0.0400,  0.0200,  0.5800],
        [ 0.4200,  0.3400,  0.2500]], grad_fn=<RoundBackward1>)

In [6]:
sparse_model.fc0.replace_many(torch.tensor([0, 1, 1]), torch.tensor([0, 0, 2]))

3


In [7]:
torch.round(sparse_model.fc0.embed_linears[0].create_sparse_tensor().to_dense(), decimals=2)

tensor([[1., 0., 0.],
        [1., 0., 0.],
        [0., 0., 1.]], grad_fn=<RoundBackward1>)

In [8]:
sparse_model.fc0.embed_linears[0].create_sparse_tensor().to_dense()

tensor([[1.0000e+00, 0.0000e+00, 4.4322e-09],
        [1.0000e+00, 0.0000e+00, 9.9591e-09],
        [3.9790e-09, 0.0000e+00, 1.0000e+00]], grad_fn=<ToDenseBackward0>)

In [9]:
sparse_model.fc0.embed_linears[0].create_sparse_tensor()

tensor(indices=tensor([[0, 1, 2, 0, 1, 2],
                       [0, 0, 2, 2, 2, 0]]),
       values=tensor([1.0000e+00, 1.0000e+00, 1.0000e+00, 4.4322e-09,
                      9.9591e-09, 3.9790e-09]),
       size=(3, 3), nnz=6, layout=torch.sparse_coo,
       grad_fn=<SparseCooTensorWithDimsAndTensorsBackward0>)

In [10]:
torch.round(sparse_model.fc0.create_sparse_tensor().to_dense(), decimals=2)

tensor([[ 0.0000, -0.4800, -0.5300, -0.5100,  0.0000,  0.0000],
        [ 0.0000,  0.0200,  0.0000,  0.0000,  0.0400,  0.5800],
        [ 0.4200,  0.3400,  0.2500,  0.0000,  0.0000,  0.0000]],
       grad_fn=<RoundBackward1>)

In [11]:
import jaxtyping
%load_ext jaxtyping
%jaxtyping.typechecker beartype.beartype

In [12]:
from ssonn.model.model import ScaledNet

In [13]:
some_data = torch.rand((1, 5, 3))

weight = torch.rand((3, 3))
bias = torch.rand((1, 3))
scaled_net = ScaledNet(weight, bias, device='cpu')

scaled_net(some_data)

tensor([[[0.7101, 0.6416, 1.6306],
         [0.9892, 0.5626, 1.7228],
         [0.3508, 0.7435, 1.4788],
         [0.5790, 0.9012, 1.8276],
         [0.6256, 0.7083, 1.6367]]], grad_fn=<ViewBackward0>)

In [14]:
scaled_net.linear.weight

Parameter containing:
tensor([[0.0583, 0.9553, 0.2164],
        [0.7348, 0.2224, 0.0468],
        [0.7161, 0.9503, 0.0207]], requires_grad=True)

In [15]:
scaled_net.put_sub_linear(torch.tensor([0, 1, 1]), torch.tensor([0, 0, 2]))

In [16]:
scaled_net.sub_linears[0].linear.weight

Parameter containing:
tensor([[1.0000e+00, 0.0000e+00, 4.4137e-10],
        [1.0000e+00, 0.0000e+00, 2.0642e-10],
        [2.8950e-10, 0.0000e+00, 1.0000e+00]], requires_grad=True)

In [17]:
scaled_net.linear.weight

Parameter containing:
tensor([[0.0000, 0.9553, 0.2164, 0.0583, 0.0000, 0.0000],
        [0.0000, 0.2224, 0.0000, 0.0000, 0.7348, 0.0468],
        [0.7161, 0.9503, 0.0207, 0.0000, 0.0000, 0.0000]], requires_grad=True)

In [18]:
scaled_net(some_data)

torch.Size([1, 5, 3])
torch.Size([1, 5, 6])


tensor([[[0.7101, 0.6416, 1.6306],
         [0.9892, 0.5626, 1.7228],
         [0.3508, 0.7435, 1.4788],
         [0.5790, 0.9012, 1.8276],
         [0.6256, 0.7083, 1.6367]]], grad_fn=<ViewBackward0>)